# Claude Advisor Tool — Workaround for Amazon Bedrock

## What is the Advisor Tool?

The [Claude Advisor Tool](https://docs.anthropic.com/en/docs/build-with-claude/advisor) is an Anthropic API feature that **pairs a strong model (Opus) as a planning advisor with a fast model (Sonnet) as the executor** — in a single API call.

The executor decides *when* to consult the advisor. When it does, the server automatically runs Opus with the full conversation transcript and returns its guidance. The executor continues generating, now informed.

```
┌─── Single /v1/messages request ───────────────────────┐
│                                                       │
│  Executor (Sonnet) ──▶ "I need guidance" ──▶ Server   │
│                                                │      │
│                                     Runs Advisor (Opus)│
│                                                │      │
│  Executor (Sonnet) ◀── advisor guidance ◀──────┘      │
│        │                                              │
│        ▼                                              │
│  Final response                                       │
└───────────────────────────────────────────────────────┘
```

### Where is it available?

| Platform | Available? |
|----------|------------|
| Anthropic API (direct) | ✅ Beta (`advisor-tool-2026-03-01`) |
| Claude Platform on AWS | ✅ |
| Amazon Bedrock (Converse / InvokeModel) | ❌ Not yet |

This notebook implements the **same pattern client-side** on Amazon Bedrock.

## Why Server-Side is Better (and why we still need this workaround)

The native advisor tool handles everything server-side. Here's why that's superior:

| | Server-Side (Native) | Client-Side (This Workaround) |
|---|---|---|
| **Latency** | Single request — advisor runs in-process with no network round-trip between models | Extra round-trip per advisor call (client → Bedrock → client → Bedrock) |
| **Simplicity** | One API call, one response | Agentic loop managing multiple calls |
| **Transcript handling** | Server automatically constructs advisor's view from full context | Client must serialize and pass the full conversation manually |
| **Caching** | Built-in ephemeral caching (`ttl: 5m`) reduces cost on repeated advisor calls | No built-in caching (would need Bedrock prompt caching separately) |
| **Token visibility** | `usage.iterations[]` gives clean per-role breakdown | Must track manually |
| **Atomicity** | If executor + advisor run in one request, no partial failure between calls | Advisor call can fail mid-loop, requiring retry logic |

### So why use client-side?

**Because Bedrock doesn't support the native tool yet.** If you're building on Bedrock (for VPC networking, AWS IAM auth, compliance, existing billing, etc.), this workaround gives you the same *pattern* and *cost profile* — you just eat the extra latency and complexity.

When Bedrock adds native advisor tool support, you'd migrate by:
1. Removing the orchestration loop
2. Adding the advisor to your tool list with `type: advisor_20260301`
3. Switching to a single `converse()` call

The architecture stays the same — only the plumbing changes.

## Setup

In [ ]:
%pip install boto3 --quiet

In [ ]:
import json
import boto3

# Configuration — adjust to your region and model access
REGION = "us-east-1"
EXECUTOR_MODEL = "us.anthropic.claude-sonnet-4-6-v1"  # Fast, cheap
ADVISOR_MODEL = "us.anthropic.claude-opus-4-7-v1"     # Strong, expensive

client = boto3.client("bedrock-runtime", region_name=REGION)
print(f"✓ Bedrock client ready ({REGION})")
print(f"  Executor: {EXECUTOR_MODEL}")
print(f"  Advisor:  {ADVISOR_MODEL}")

## Step 1: Define the Advisor Tool

We give the executor a tool called `consult_advisor`. The tool description tells the model **when** to use it — following Anthropic's own guidance:

1. **Before substantive work** (not orientation — actual writing/building)
2. **When stuck** (recurring errors, approach not converging)
3. **Before declaring done** (final review)
4. **On disagreement** (when evidence contradicts prior advisor guidance)

In [ ]:
ADVISOR_TOOL = {
    "toolSpec": {
        "name": "consult_advisor",
        "description": (
            "Consult a senior advisor for strategic guidance. The advisor sees "
            "the full conversation and provides expert-level review. "
            "Call this: (1) before substantive work begins, "
            "(2) when stuck or an approach isn't converging, "
            "(3) before declaring done for a final review, "
            "(4) when your evidence contradicts prior advice — surface the conflict."
        ),
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "What specific guidance do you need?"
                    }
                },
                "required": ["question"]
            }
        }
    }
}

print("✓ Advisor tool defined")
print(json.dumps(ADVISOR_TOOL, indent=2))

## Step 2: Define System Prompts

Two models, two roles, two prompts.

In [ ]:
EXECUTOR_SYSTEM = (
    "You are a capable AI assistant. You have access to a senior advisor "
    "via the consult_advisor tool. Use it for complex decisions — don't "
    "consult for trivial questions, but do consult before major design "
    "decisions or when you're unsure about the best approach."
)

ADVISOR_SYSTEM = (
    "You are a senior technical advisor. Review the executor's work and "
    "provide concise, actionable guidance. Focus on: correctness, approach "
    "quality, edge cases missed, and strategic direction. "
    "Keep responses under 500 words. Be direct."
)

## Step 3: The Orchestration Loop

This is the core workaround. The loop:
1. Calls the executor (Sonnet) with the conversation + advisor tool
2. If the executor calls `consult_advisor`, intercepts and routes to the advisor (Opus)
3. Returns the advisor's guidance as a tool result
4. Repeats until the executor produces a final response

In [ ]:
def run_advisor_pattern(user_message: str, max_advisor_calls: int = 3) -> dict:
    """
    Run the advisor-executor pattern on Bedrock.
    
    Returns dict with 'response', 'usage', and 'advisor_calls'.
    """
    messages = [{"role": "user", "content": [{"text": user_message}]}]
    usage = {"executor": {"in": 0, "out": 0}, "advisor": {"in": 0, "out": 0}}
    advisor_calls = 0

    while True:
        # --- Call Executor (Sonnet) ---
        response = client.converse(
            modelId=EXECUTOR_MODEL,
            messages=messages,
            system=[{"text": EXECUTOR_SYSTEM}],
            toolConfig={"tools": [ADVISOR_TOOL]},
            inferenceConfig={"maxTokens": 4096},
        )

        # Track executor usage
        u = response.get("usage", {})
        usage["executor"]["in"] += u.get("inputTokens", 0)
        usage["executor"]["out"] += u.get("outputTokens", 0)

        # Add assistant message to history
        assistant_msg = response["output"]["message"]
        messages.append(assistant_msg)

        # If executor finished naturally, we're done
        if response["stopReason"] == "end_turn":
            break

        # If executor wants to use a tool
        if response["stopReason"] == "tool_use":
            tool_results = []

            for block in assistant_msg.get("content", []):
                if "toolUse" not in block:
                    continue

                tool = block["toolUse"]
                if tool["name"] != "consult_advisor":
                    continue

                # --- Call Advisor (Opus) ---
                if advisor_calls >= max_advisor_calls:
                    advice = "[Advisor limit reached. Proceed with your best judgment.]"
                else:
                    question = tool["input"].get("question", "")
                    print(f"  🧠 Advisor consulted: {question[:80]}...")

                    advisor_response = client.converse(
                        modelId=ADVISOR_MODEL,
                        messages=[{
                            "role": "user",
                            "content": [{
                                "text": (
                                    f"## Full conversation so far\n\n"
                                    f"{json.dumps(messages, indent=2, default=str)}\n\n"
                                    f"---\n\n"
                                    f"## Executor's question\n{question}"
                                )
                            }]
                        }],
                        system=[{"text": ADVISOR_SYSTEM}],
                        inferenceConfig={"maxTokens": 1024},
                    )

                    # Track advisor usage
                    au = advisor_response.get("usage", {})
                    usage["advisor"]["in"] += au.get("inputTokens", 0)
                    usage["advisor"]["out"] += au.get("outputTokens", 0)
                    advisor_calls += 1

                    # Extract advisor text
                    advice = ""
                    for b in advisor_response["output"]["message"]["content"]:
                        if "text" in b:
                            advice += b["text"]

                    print(f"  ✓ Advisor responded ({len(advice)} chars)")

                tool_results.append({
                    "toolResult": {
                        "toolUseId": tool["toolUseId"],
                        "content": [{"text": advice}]
                    }
                })

            # Return advisor guidance to executor
            messages.append({"role": "user", "content": tool_results})
        else:
            break  # Unknown stop reason

    # Extract final text
    final_text = ""
    for block in assistant_msg.get("content", []):
        if "text" in block:
            final_text += block["text"]

    return {
        "response": final_text,
        "usage": usage,
        "advisor_calls": advisor_calls,
    }

print("✓ Orchestration function defined")

## Step 4: Run It

Let's give the executor a complex task where consulting an advisor adds real value.

In [ ]:
task = """
Design a serverless event-driven architecture for a real-time fraud 
detection system processing 10,000 transactions per second.

Requirements:
- Sub-100ms latency for scoring
- ML model inference (XGBoost + neural network ensemble)
- Real-time feature engineering from streaming data
- Explainability for flagged transactions
- 99.99% availability

Provide the architecture with AWS services, data flow, and key decisions.
"""

print("🚀 Running advisor-executor pattern...\n")
result = run_advisor_pattern(task)

print(f"\n{'─' * 60}")
print(result["response"])

## Step 5: Check the Cost Split

The whole point of the advisor pattern is **cost efficiency**: Opus only for the ~400-700 token planning moments, Sonnet for the bulk generation.

In [ ]:
print("─── Token Usage ────────────────────────────────────────")
print(f"  Executor (Sonnet):  in={result['usage']['executor']['in']:,}  out={result['usage']['executor']['out']:,}")
print(f"  Advisor  (Opus):    in={result['usage']['advisor']['in']:,}  out={result['usage']['advisor']['out']:,}")
print(f"  Advisor calls:      {result['advisor_calls']}")
print("────────────────────────────────────────────────────────")

# Rough cost estimate (us-east-1 pricing as of May 2026)
SONNET_IN = 3.00 / 1_000_000   # $3/M input tokens
SONNET_OUT = 15.00 / 1_000_000  # $15/M output tokens
OPUS_IN = 15.00 / 1_000_000    # $15/M input tokens
OPUS_OUT = 75.00 / 1_000_000   # $75/M output tokens

executor_cost = (
    result['usage']['executor']['in'] * SONNET_IN +
    result['usage']['executor']['out'] * SONNET_OUT
)
advisor_cost = (
    result['usage']['advisor']['in'] * OPUS_IN +
    result['usage']['advisor']['out'] * OPUS_OUT
)

print(f"\n  💰 Estimated cost:")
print(f"     Executor: ${executor_cost:.4f}")
print(f"     Advisor:  ${advisor_cost:.4f}")
print(f"     Total:    ${executor_cost + advisor_cost:.4f}")
print(f"\n  vs. all-Opus: ${(result['usage']['executor']['in'] + result['usage']['advisor']['in']) * OPUS_IN + (result['usage']['executor']['out'] + result['usage']['advisor']['out']) * OPUS_OUT:.4f}")

## Summary: When to Use This

### Use this workaround when:
- You're on **Amazon Bedrock** (not Claude Platform on AWS or Anthropic direct)
- You need the **planning quality of Opus** without paying Opus rates for everything
- Your tasks benefit from **two-stage thinking** (plan → execute)

### Migrate to native when:
- Bedrock adds `advisor_20260301` tool type support
- Then: delete the loop, add advisor to tool list, single `converse()` call

### The pattern works because:
- Planning and execution are **qualitatively different** — planning needs more model capability per token
- The executor (Sonnet) is smart enough to know **when** it needs help
- You only pay Opus rates on the advisor's ~400-700 token responses, not the full generation